# Train, val, test split

In [ ]:
import tensorflow as tf
import tensorflow_probability as tfp
import numpy as np
import matplotlib.pyplot as plt
import os
from PIL import Image
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
import random
import json
import shutil
from collections import Counter
import optuna

In [ ]:
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
DATA_DIR = '/kaggle/input/datasets/piccini/wikiart-clean/wikiart_clean'

In [ ]:
ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    image_size=(224, 224),
    batch_size=32,
    label_mode="categorical",
    shuffle=True,
    seed=42,
    validation_split=0.3,
    subset="both",
)


In [ ]:
train_ds, val_ds = ds

val_batches = val_ds.cardinality().numpy()
test_ds = val_ds.take(val_batches // 2)
val_ds  = val_ds.skip(val_batches // 2)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(AUTOTUNE)
val_ds   = val_ds.cache().prefetch(AUTOTUNE)
test_ds  = test_ds.cache().prefetch(AUTOTUNE)

# Verify splits
print(f"Train batches : {train_ds.cardinality().numpy()}")
print(f"Val batches   : {val_ds.cardinality().numpy()}")
print(f"Test batches  : {test_ds.cardinality().numpy()}")

# Verify class names
print(f"\nClasses : {ds[0].class_names}")

In [ ]:
def check_split_reproducibility(ds_split, split_name, class_names, n=5):
    images, labels = next(iter(ds_split))
    print(f"\n{split_name.upper()} first {n} images:")
    print(f"{'Index':<8} {'Class':<30} {'Mean':>10}")
    print("-" * 50)
    for i in range(n):
        cls = class_names[tf.argmax(labels[i]).numpy()]
        mean = images[i].numpy().mean()
        print(f"{i:<8} {cls:<30} {mean:>10.4f}")

class_names = ds[0].class_names

check_split_reproducibility(train_ds, "train", class_names)
check_split_reproducibility(val_ds,   "val",   class_names)
check_split_reproducibility(test_ds,  "test",  class_names)

In [ ]:
class_names = sorted([
    d for d in os.listdir(DATA_DIR)
    if os.path.isdir(os.path.join(DATA_DIR, d))
])


In [ ]:
def get_class_distribution(dataset, class_names):
    counts = Counter()
    for _, labels in dataset:
        indices = tf.argmax(labels, axis=1).numpy()
        for idx in indices:
            counts[class_names[idx]] += 1
    return dict(sorted(counts.items()))

train_dist = get_class_distribution(train_ds, class_names)
val_dist   = get_class_distribution(val_ds, class_names)
test_dist  = get_class_distribution(test_ds, class_names)

print("=== Class Distribution ===\n")
print(f"{'Class':<30} {'Train':>8} {'Val':>8} {'Test':>8} {'Total':>8}")
print("-" * 60)
for cls in class_names:
    tr = train_dist.get(cls, 0)
    v  = val_dist.get(cls, 0)
    te = test_dist.get(cls, 0)
    print(f"{cls:<30} {tr:>8} {v:>8} {te:>8} {tr+v+te:>8}")

print("-" * 60)
total_tr = sum(train_dist.values())
total_v  = sum(val_dist.values())
total_te = sum(test_dist.values())
print(f"{'TOTAL':<30} {total_tr:>8} {total_v:>8} {total_te:>8} {total_tr+total_v+total_te:>8}")

In [ ]:
all_labels_int = []
for i, cls in enumerate(class_names):
    cls_path = os.path.join(DATA_DIR, cls)
    n_images = len(os.listdir(cls_path))
    all_labels_int.extend([i] * n_images)

all_labels_int = np.array(all_labels_int)

class_weights_values = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(all_labels_int),
    y=all_labels_int,
)

class_weight_dict = dict(enumerate(class_weights_values))
print("Class weights:")
for i, cls in enumerate(class_names):
    print(f"  {cls:<30} {class_weight_dict[i]:.4f}")

# CNN from scratch

In [ ]:
def build_cnn_scratch(
    augment_layer,
    num_classes: int = 23,
    name: str = "cnn_scratch",
) -> keras.Model:
    """
    Simple 4-block CNN trained from scratch.
    Each block: Conv → BN → ReLU → Conv → BN → ReLU → MaxPool → Dropout
    Keeps capacity moderate — a huge scratch model will just overfit
    and tell you nothing useful.
    """
    inputs = keras.Input(shape=(224, 224, 3))
    x = augment_layer(inputs)


    x = layers.Rescaling(1.0 / 255)(x)

    x = layers.Conv2D(32, 3, padding="same", kernel_regularizer=keras.regularizers.L2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Conv2D(32, 3, padding="same", kernel_regularizer=keras.regularizers.L2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Dropout(0.2)(x)

    x = layers.Conv2D(64, 3, padding="same", kernel_regularizer=keras.regularizers.L2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Conv2D(64, 3, padding="same", kernel_regularizer=keras.regularizers.L2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Dropout(0.2)(x)

    x = layers.Conv2D(128, 3, padding="same", kernel_regularizer=keras.regularizers.L2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Conv2D(128, 3, padding="same", kernel_regularizer=keras.regularizers.L2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Conv2D(256, 3, padding="same", kernel_regularizer=keras.regularizers.L2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Conv2D(256, 3, padding="same", kernel_regularizer=keras.regularizers.L2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.4)(x)

    x = layers.Dense(256, activation="relu",
                     kernel_regularizer=keras.regularizers.L2(1e-4))(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    return keras.Model(inputs, outputs, name=name)

In [ ]:
augment_none = keras.layers.Lambda(lambda x: x, name="augmentation_none")
model_scratch = build_cnn_scratch(augment_none, name="cnn_scratch_noaug")

model_scratch.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=[
        keras.metrics.F1Score(average="macro", name="f1_score"),
        keras.metrics.TopKCategoricalAccuracy(k=3, name="top3_acc"),
    ],
)

history_scratch = model_scratch.fit(
    train_ds,
    validation_data=val_ds,
    epochs=60,              
    class_weight=class_weight_dict,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor="val_f1_score", patience=8,
            restore_best_weights=True, mode="max",
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_f1_score", factor=0.5,
            patience=4, min_lr=1e-6, mode="max", verbose=1,
        ),
        keras.callbacks.ModelCheckpoint(
            filepath="/kaggle/working/models/cnn_scratch_noaug.keras",
            monitor="val_f1_score", save_best_only=True,
            mode="max", verbose=1,
        ),
    ],
    verbose=1,
)

best_epoch  = int(np.argmax(history_scratch.history["val_f1_score"]))
val_f1      = float(history_scratch.history["val_f1_score"][best_epoch])
train_f1    = float(history_scratch.history["f1_score"][best_epoch])
stopped     = len(history_scratch.history["val_f1_score"])

print(f"\n  CNN from scratch")
print(f"  Val F1      : {val_f1:.4f}")
print(f"  Train F1    : {train_f1:.4f}")
print(f"  Gap         : {train_f1 - val_f1:.4f}")
print(f"  Stopped     : epoch {stopped}")

In [ ]:
augment_light = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
], name="aug_light")

model_scratch = build_cnn_scratch(augment_light, name="cnn_scratch_laug")

model_scratch.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=[
        keras.metrics.F1Score(average="macro", name="f1_score"),
        keras.metrics.TopKCategoricalAccuracy(k=3, name="top3_acc"),
    ],
)

history_scratch = model_scratch.fit(
    train_ds,
    validation_data=val_ds,
    epochs=60,             
    class_weight=class_weight_dict,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor="val_f1_score", patience=8,
            restore_best_weights=True, mode="max",
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_f1_score", factor=0.5,
            patience=4, min_lr=1e-6, mode="max", verbose=1,
        ),
        keras.callbacks.ModelCheckpoint(
            filepath="/kaggle/working/models/cnn_scratch_best.keras",
            monitor="val_f1_score", save_best_only=True,
            mode="max", verbose=1,
        ),
    ],
    verbose=1,
)

best_epoch  = int(np.argmax(history_scratch.history["val_f1_score"]))
val_f1      = float(history_scratch.history["val_f1_score"][best_epoch])
train_f1    = float(history_scratch.history["f1_score"][best_epoch])
stopped     = len(history_scratch.history["val_f1_score"])

print(f"\n  CNN from scratch")
print(f"  Val F1      : {val_f1:.4f}")
print(f"  Train F1    : {train_f1:.4f}")
print(f"  Gap         : {train_f1 - val_f1:.4f}")
print(f"  Stopped     : epoch {stopped}")

In [ ]:
augment_moderate = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomBrightness(0.1),
    layers.RandomContrast(0.1),
], name="aug_moderate")

model_scratch = build_cnn_scratch(augment_moderate, name="cnn_scratch_caug")

model_scratch.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=[
        keras.metrics.F1Score(average="macro", name="f1_score"),
        keras.metrics.TopKCategoricalAccuracy(k=3, name="top3_acc"),
    ],
)

history_scratch = model_scratch.fit(
    train_ds,
    validation_data=val_ds,
    epochs=60,             
    class_weight=class_weight_dict,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor="val_f1_score", patience=8,
            restore_best_weights=True, mode="max",
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_f1_score", factor=0.5,
            patience=4, min_lr=1e-6, mode="max", verbose=1,
        ),
        keras.callbacks.ModelCheckpoint(
            filepath="/kaggle/working/models/cnn_scratch_caug.keras",
            monitor="val_f1_score", save_best_only=True,
            mode="max", verbose=1,
        ),
    ],
    verbose=1,
)

best_epoch  = int(np.argmax(history_scratch.history["val_f1_score"]))
val_f1      = float(history_scratch.history["val_f1_score"][best_epoch])
train_f1    = float(history_scratch.history["f1_score"][best_epoch])
stopped     = len(history_scratch.history["val_f1_score"])

print(f"\n  CNN from scratch")
print(f"  Val F1      : {val_f1:.4f}")
print(f"  Train F1    : {train_f1:.4f}")
print(f"  Gap         : {train_f1 - val_f1:.4f}")
print(f"  Stopped     : epoch {stopped}")

# Pre-Trained Models

## EfficientNetV2s

In [ ]:
def build_efficientnet_model(
    augment_layer,
    num_classes: int = 23,
    name: str = "efficientnetv2s",
) -> keras.Model:
    inputs = keras.Input(shape=(224, 224, 3))
    x = augment_layer(inputs)
    
    base = keras.applications.EfficientNetV2S(
        include_top=False,
        weights="imagenet",
        input_tensor=x,
        pooling="avg", 
    )
    base.trainable = False  
    
    x = base.output
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(256, activation="relu",
                     kernel_regularizer=keras.regularizers.L2(1e-4))(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    
    return keras.Model(inputs, outputs, name=name)

augment_phase1 = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomBrightness(0.1),
    layers.RandomContrast(0.1),
], name="aug_phase1")

model_eff = build_efficientnet_model(augment_phase1, name="efficientnetv2s")

model_eff.compile(
    optimizer=keras.optimizers.Adam(1e-3),  # high LR is fine, base is frozen
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=[
        keras.metrics.F1Score(average="macro", name="f1_score"),
        keras.metrics.TopKCategoricalAccuracy(k=3, name="top3_acc"),
    ],
)

history_phase1 = model_eff.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,  
    class_weight=class_weight_dict,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor="val_f1_score", patience=5,
            restore_best_weights=True, mode="max",
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_f1_score", factor=0.5,
            patience=3, min_lr=1e-6, mode="max", verbose=1,
        ),
        keras.callbacks.ModelCheckpoint(
            filepath="/kaggle/working/models/efficientnetv2s_phase1.keras",
            monitor="val_f1_score", save_best_only=True,
            mode="max", verbose=1,
        ),
    ],
    verbose=1,
)

In [ ]:
for i, layer in enumerate(model_eff.layers):
    print(f"{i:>3d}  {layer.name}  —  {type(layer).__name__}")

In [ ]:
for i, layer in enumerate(model_eff.layers):
    if i < 288:
        layer.trainable = False
    else:
        layer.trainable = True


trainable = sum(1 for l in model_eff.layers if l.trainable)
frozen = sum(1 for l in model_eff.layers if not l.trainable)
print(f"Trainable: {trainable}, Frozen: {frozen}")

In [ ]:
model_eff.compile(
    optimizer=keras.optimizers.Adam(1e-4),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=[
        keras.metrics.F1Score(average="macro", name="f1_score"),
        keras.metrics.TopKCategoricalAccuracy(k=3, name="top3_acc"),
    ],
)

history_phase2 = model_eff.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    class_weight=class_weight_dict,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor="val_f1_score", patience=7,
            restore_best_weights=True, mode="max",
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_f1_score", factor=0.5,
            patience=3, min_lr=1e-7, mode="max", verbose=1,
        ),
        keras.callbacks.ModelCheckpoint(
            filepath="/kaggle/working/models/efficientnetv2s_phase2.keras",
            monitor="val_f1_score", save_best_only=True,
            mode="max", verbose=1,
        ),
    ],
    verbose=1,
)

In [ ]:
for i, layer in enumerate(model_eff.layers):
    if i < 155:
        layer.trainable = False
    else:
        layer.trainable = True

# Verify
trainable = sum(1 for l in model_eff.layers if l.trainable)
frozen = sum(1 for l in model_eff.layers if not l.trainable)
print(f"Trainable: {trainable}, Frozen: {frozen}")

In [ ]:
model_eff.compile(
    optimizer=keras.optimizers.Adam(5e-5),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=[
        keras.metrics.F1Score(average="macro", name="f1_score"),
        keras.metrics.TopKCategoricalAccuracy(k=3, name="top3_acc"),
    ],
)

history_phase3 = model_eff.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    class_weight=class_weight_dict,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor="val_f1_score", patience=7,
            restore_best_weights=True, mode="max",
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_f1_score", factor=0.5,
            patience=3, min_lr=1e-7, mode="max", verbose=1,
        ),
        keras.callbacks.ModelCheckpoint(
            filepath="/kaggle/working/models/efficientnetv2s_phase3.keras",
            monitor="val_f1_score", save_best_only=True,
            mode="max", verbose=1,
        ),
    ],
    verbose=1,
)

best_epoch = int(np.argmax(history_phase3.history["val_f1_score"]))
val_f1     = float(history_phase3.history["val_f1_score"][best_epoch])
train_f1   = float(history_phase3.history["f1_score"][best_epoch])
stopped    = len(history_phase3.history["val_f1_score"])

print(f"\n  EfficientNetV2S Phase 3")
print(f"  Val F1      : {val_f1:.4f}")
print(f"  Train F1    : {train_f1:.4f}")
print(f"  Gap         : {train_f1 - val_f1:.4f}")
print(f"  Stopped     : epoch {stopped}")

In [ ]:
all_preds = []
all_labels = []

for images, labels in val_ds:
    preds = model_eff(images, training=False)
    all_preds.append(preds.numpy())
    all_labels.append(labels.numpy())

all_preds  = np.concatenate(all_preds, axis=0)
all_labels = np.concatenate(all_labels, axis=0)

pred_classes  = np.argmax(all_preds, axis=1)
label_classes = np.argmax(all_labels, axis=1)


print(classification_report(label_classes, pred_classes, target_names=class_names, digits=3))

In [ ]:
cm = confusion_matrix(label_classes, pred_classes)

weak_indices = [
    class_names.index(c) for c in [
        "Salvador_Dali", "Boris_Kustodiev", 
        "Ilya_Repin", "Pablo_Picasso", "Paul_Cezanne"
    ]
]

for idx in weak_indices:
    row = cm[idx]
    top_confusions = np.argsort(row)[::-1][:4]
    print(f"\n{class_names[idx]} (true label) predicted as:")
    for pred_idx in top_confusions:
        if row[pred_idx] > 0:
            print(f"  {class_names[pred_idx]:<30} {row[pred_idx]:>3} times")

In [ ]:
for i, layer in enumerate(model_eff.layers):
    print(f"[{i}] {layer.name}")

In [ ]:
for i, layer in enumerate(model_v2s_v2.layers):
    print(f"{i:>3d}  {layer.name}  —  {type(layer).__name__}")

In [ ]:
def build_efficientnetv2s_v2(
    augment_layer,
    num_classes: int = 23,
    name: str = "efficientnetv2s_v2",
) -> keras.Model:
    inputs = keras.Input(shape=(224, 224, 3))
    x = augment_layer(inputs)

    base = keras.applications.EfficientNetV2S(
        include_top=False,
        weights="imagenet",
        input_tensor=x,
        pooling="avg",
    )
    base.trainable = False

    for layer in base.layers:
        if isinstance(layer, keras.layers.BatchNormalization):
            layer.trainable = False

    x = base.output
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(256, activation="relu",
                     kernel_regularizer=keras.regularizers.L2(3e-4))(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    return keras.Model(inputs, outputs, name=name)

augment_v2s_v2 = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomBrightness(0.1),
    layers.RandomContrast(0.1),
], name="aug_v2s_v2")

# Load best phase 1 weights
model_v2s_v2 = build_efficientnetv2s_v2(augment_v2s_v2, name="efficientnetv2s_v2")
model_v2s_v2.load_weights("/kaggle/working/models/efficientnetv2s_v2_phase1.keras")
print("Phase 1 weights loaded")

In [ ]:
for i, layer in enumerate(model_v2s_v2.layers[460:520]):
    print(f"{460+i:>3d}  {layer.name}")

In [ ]:
def build_efficientnetv2s_v2(
    augment_layer,
    num_classes: int = 23,
    name: str = "efficientnetv2s_v2",
) -> keras.Model:
    inputs = keras.Input(shape=(224, 224, 3))
    x = augment_layer(inputs)

    base = keras.applications.EfficientNetV2S(
        include_top=False,
        weights="imagenet",
        input_tensor=x,
        pooling="avg",
    )
    base.trainable = False

    # Freeze BN layers permanently across all phases
    for layer in base.layers:
        if isinstance(layer, keras.layers.BatchNormalization):
            layer.trainable = False

    x = base.output
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(256, activation="relu",
                     kernel_regularizer=keras.regularizers.L2(3e-4))(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    return keras.Model(inputs, outputs, name=name)


augment_v2s_v2 = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomBrightness(0.1),
    layers.RandomContrast(0.1),
], name="aug_v2s_v2")

model_v2s_v2 = build_efficientnetv2s_v2(augment_v2s_v2, name="efficientnetv2s_v2")

model_v2s_v2.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=[
        keras.metrics.F1Score(average="macro", name="f1_score"),
        keras.metrics.TopKCategoricalAccuracy(k=3, name="top3_acc"),
    ],
)

history_v2s_v2_phase1 = model_v2s_v2.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    class_weight=class_weight_dict,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor="val_f1_score", patience=5,
            restore_best_weights=True, mode="max",
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_f1_score", factor=0.5,
            patience=3, min_lr=1e-6, mode="max", verbose=1,
        ),
        keras.callbacks.ModelCheckpoint(
            filepath="/kaggle/working/models/efficientnetv2s_v2_phase1.keras",
            monitor="val_f1_score", save_best_only=True,
            mode="max", verbose=1,
        ),
    ],
    verbose=1,
)

best_epoch = int(np.argmax(history_v2s_v2_phase1.history["val_f1_score"]))
val_f1     = float(history_v2s_v2_phase1.history["val_f1_score"][best_epoch])
train_f1   = float(history_v2s_v2_phase1.history["f1_score"][best_epoch])
stopped    = len(history_v2s_v2_phase1.history["val_f1_score"])

print(f"\n  EfficientNetV2S v2 Phase 1")
print(f"  Val F1      : {val_f1:.4f}")
print(f"  Train F1    : {train_f1:.4f}")
print(f"  Gap         : {train_f1 - val_f1:.4f}")
print(f"  Stopped     : epoch {stopped}")

In [ ]:
history_v2s_v2_phase1b = model_v2s_v2.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,  
    class_weight=class_weight_dict,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor="val_f1_score", patience=5,
            restore_best_weights=True, mode="max",
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_f1_score", factor=0.5,
            patience=3, min_lr=1e-6, mode="max", verbose=1,
        ),
        keras.callbacks.ModelCheckpoint(
            filepath="/kaggle/working/models/efficientnetv2s_v2_phase1.keras",
            monitor="val_f1_score", save_best_only=True,
            mode="max", verbose=1,
        ),
    ],
    verbose=1,
)

best_epoch = int(np.argmax(history_v2s_v2_phase1b.history["val_f1_score"]))
val_f1     = float(history_v2s_v2_phase1b.history["val_f1_score"][best_epoch])
train_f1   = float(history_v2s_v2_phase1b.history["f1_score"][best_epoch])
stopped    = len(history_v2s_v2_phase1b.history["val_f1_score"])

print(f"\n  EfficientNetV2S v2 Phase 1b")
print(f"  Val F1      : {val_f1:.4f}")
print(f"  Train F1    : {train_f1:.4f}")
print(f"  Gap         : {train_f1 - val_f1:.4f}")
print(f"  Stopped     : epoch {stopped}")

In [ ]:
history_v2s_v2_phase1c = model_v2s_v2.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    class_weight=class_weight_dict,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor="val_f1_score", patience=5,
            restore_best_weights=True, mode="max",
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_f1_score", factor=0.5,
            patience=3, min_lr=1e-6, mode="max", verbose=1,
        ),
        keras.callbacks.ModelCheckpoint(
            filepath="/kaggle/working/models/efficientnetv2s_v2_phase1.keras",
            monitor="val_f1_score", save_best_only=True,
            mode="max", verbose=1,
        ),
    ],
    verbose=1,
)

best_epoch = int(np.argmax(history_v2s_v2_phase1c.history["val_f1_score"]))
val_f1     = float(history_v2s_v2_phase1c.history["val_f1_score"][best_epoch])
train_f1   = float(history_v2s_v2_phase1c.history["f1_score"][best_epoch])
stopped    = len(history_v2s_v2_phase1c.history["val_f1_score"])

print(f"\n  EfficientNetV2S v2 Phase 1c")
print(f"  Val F1      : {val_f1:.4f}")
print(f"  Train F1    : {train_f1:.4f}")
print(f"  Gap         : {train_f1 - val_f1:.4f}")
print(f"  Stopped     : epoch {stopped}")

In [ ]:
for i, layer in enumerate(model_v2s_v2.layers):
    if i < 466:
        layer.trainable = False
    else:
        layer.trainable = True

# Keep ALL BN layers frozen
for layer in model_v2s_v2.layers:
    if isinstance(layer, keras.layers.BatchNormalization):
        layer.trainable = False

trainable = sum(1 for l in model_v2s_v2.layers if l.trainable)
frozen = sum(1 for l in model_v2s_v2.layers if not l.trainable)
print(f"Trainable: {trainable}, Frozen: {frozen}")

model_v2s_v2.compile(
    optimizer=keras.optimizers.Adam(1e-5),  # lower than before
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=[
        keras.metrics.F1Score(average="macro", name="f1_score"),
        keras.metrics.TopKCategoricalAccuracy(k=3, name="top3_acc"),
    ],
)

history_v2s_v2_phase2 = model_v2s_v2.fit(
    train_ds,
    validation_data=val_ds,
    epochs=40,
    class_weight=class_weight_dict,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor="val_f1_score", patience=8,
            restore_best_weights=True, mode="max",
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_f1_score", factor=0.5,
            patience=4, min_lr=1e-8, mode="max", verbose=1,
        ),
        keras.callbacks.ModelCheckpoint(
            filepath="/kaggle/working/models/efficientnetv2s_v2_phase2.keras",
            monitor="val_f1_score", save_best_only=True,
            mode="max", verbose=1,
        ),
    ],
    verbose=1,
)

best_epoch = int(np.argmax(history_v2s_v2_phase2.history["val_f1_score"]))
val_f1     = float(history_v2s_v2_phase2.history["val_f1_score"][best_epoch])
train_f1   = float(history_v2s_v2_phase2.history["f1_score"][best_epoch])
stopped    = len(history_v2s_v2_phase2.history["val_f1_score"])

print(f"\n  EfficientNetV2S v2 Phase 2 (minimal unfreeze)")
print(f"  Val F1      : {val_f1:.4f}")
print(f"  Train F1    : {train_f1:.4f}")
print(f"  Gap         : {train_f1 - val_f1:.4f}")
print(f"  Stopped     : epoch {stopped}")

## MobileNet

In [ ]:
def build_mobilenetv3(
    augment_layer,
    num_classes: int = 23,
    name: str = "mobilenetv3",
) -> keras.Model:
    inputs = keras.Input(shape=(224, 224, 3))
    x = augment_layer(inputs)

    base = keras.applications.MobileNetV3Large(
        include_top=False,
        weights="imagenet",
        input_tensor=x,
        pooling="avg",
        include_preprocessing=True,  
    )
    base.trainable = False

    # Freeze BN layers
    for layer in base.layers:
        if isinstance(layer, keras.layers.BatchNormalization):
            layer.trainable = False

    x = base.output
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(256, activation="relu",
                     kernel_regularizer=keras.regularizers.L2(3e-4))(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    return keras.Model(inputs, outputs, name=name)


augment_mob = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomBrightness(0.1),
    layers.RandomContrast(0.1),
], name="aug_mob")

model_mob = build_mobilenetv3(augment_mob, name="mobilenetv3")

model_mob.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=[
        keras.metrics.F1Score(average="macro", name="f1_score"),
        keras.metrics.TopKCategoricalAccuracy(k=3, name="top3_acc"),
    ],
)

history_mob_phase1 = model_mob.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    class_weight=class_weight_dict,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor="val_f1_score", patience=7,
            restore_best_weights=True, mode="max",
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_f1_score", factor=0.5,
            patience=4, min_lr=1e-6, mode="max", verbose=1,
        ),
        keras.callbacks.ModelCheckpoint(
            filepath="/kaggle/working/models/mobilenetv3_phase1.keras",
            monitor="val_f1_score", save_best_only=True,
            mode="max", verbose=1,
        ),
    ],
    verbose=1,
)

best_epoch = int(np.argmax(history_mob_phase1.history["val_f1_score"]))
val_f1     = float(history_mob_phase1.history["val_f1_score"][best_epoch])
train_f1   = float(history_mob_phase1.history["f1_score"][best_epoch])
stopped    = len(history_mob_phase1.history["val_f1_score"])

print(f"\n  MobileNetV3Large Phase 1")
print(f"  Val F1      : {val_f1:.4f}")
print(f"  Train F1    : {train_f1:.4f}")
print(f"  Gap         : {train_f1 - val_f1:.4f}")
print(f"  Stopped     : epoch {stopped}")

In [ ]:
for i, layer in enumerate(model_mob.layers):
    print(f"{i:>3d}  {layer.name}  —  {type(layer).__name__}")

In [ ]:
for i, layer in enumerate(model_mob.layers):
    if i < 140:
        layer.trainable = False
    else:
        layer.trainable = True

# Keep BN frozen
for layer in model_mob.layers:
    if isinstance(layer, keras.layers.BatchNormalization):
        layer.trainable = False

trainable = sum(1 for l in model_mob.layers if l.trainable)
frozen = sum(1 for l in model_mob.layers if not l.trainable)
print(f"Trainable: {trainable}, Frozen: {frozen}")

model_mob.compile(
    optimizer=keras.optimizers.Adam(1e-4),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=[
        keras.metrics.F1Score(average="macro", name="f1_score"),
        keras.metrics.TopKCategoricalAccuracy(k=3, name="top3_acc"),
    ],
)

history_mob_phase2 = model_mob.fit(
    train_ds,
    validation_data=val_ds,
    epochs=40,
    class_weight=class_weight_dict,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor="val_f1_score", patience=8,
            restore_best_weights=True, mode="max",
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_f1_score", factor=0.5,
            patience=4, min_lr=1e-7, mode="max", verbose=1,
        ),
        keras.callbacks.ModelCheckpoint(
            filepath="/kaggle/working/models/mobilenetv3_phase2.keras",
            monitor="val_f1_score", save_best_only=True,
            mode="max", verbose=1,
        ),
    ],
    verbose=1,
)

best_epoch = int(np.argmax(history_mob_phase2.history["val_f1_score"]))
val_f1     = float(history_mob_phase2.history["val_f1_score"][best_epoch])
train_f1   = float(history_mob_phase2.history["f1_score"][best_epoch])
stopped    = len(history_mob_phase2.history["val_f1_score"])

print(f"\n  MobileNetV3Large Phase 2")
print(f"  Val F1      : {val_f1:.4f}")
print(f"  Train F1    : {train_f1:.4f}")
print(f"  Gap         : {train_f1 - val_f1:.4f}")
print(f"  Stopped     : epoch {stopped}")

In [ ]:
def build_mobilenetv3(
    augment_layer,
    num_classes: int = 23,
    name: str = "mobilenetv3",
) -> keras.Model:
    inputs = keras.Input(shape=(224, 224, 3))
    x = augment_layer(inputs)

    base = keras.applications.MobileNetV3Large(
        include_top=False,
        weights="imagenet",
        input_tensor=x,
        pooling="avg",
        include_preprocessing=True,
    )
    base.trainable = False

    for layer in base.layers:
        if isinstance(layer, keras.layers.BatchNormalization):
            layer.trainable = False

    x = base.output
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(256, activation="relu",
                     kernel_regularizer=keras.regularizers.L2(3e-4))(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    return keras.Model(inputs, outputs, name=name)


augment_mob2 = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomBrightness(0.1),
    layers.RandomContrast(0.1),
], name="aug_mob2")

model_mob2 = build_mobilenetv3(augment_mob2, name="mobilenetv3_v2")

model_mob2.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=[
        keras.metrics.F1Score(average="macro", name="f1_score"),
        keras.metrics.TopKCategoricalAccuracy(k=3, name="top3_acc"),
    ],
)

history_mob2_phase1 = model_mob2.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    class_weight=class_weight_dict,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor="val_f1_score", patience=7,
            restore_best_weights=True, mode="max",
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_f1_score", factor=0.5,
            patience=4, min_lr=1e-6, mode="max", verbose=1,
        ),
        keras.callbacks.ModelCheckpoint(
            filepath="/kaggle/working/models/mobilenetv3_v2_phase1.keras",
            monitor="val_f1_score", save_best_only=True,
            mode="max", verbose=1,
        ),
    ],
    verbose=1,
)

best_epoch = int(np.argmax(history_mob2_phase1.history["val_f1_score"]))
val_f1     = float(history_mob2_phase1.history["val_f1_score"][best_epoch])
train_f1   = float(history_mob2_phase1.history["f1_score"][best_epoch])
stopped    = len(history_mob2_phase1.history["val_f1_score"])

print(f"\n  MobileNetV3Large v2 Phase 1")
print(f"  Val F1      : {val_f1:.4f}")
print(f"  Train F1    : {train_f1:.4f}")
print(f"  Gap         : {train_f1 - val_f1:.4f}")
print(f"  Stopped     : epoch {stopped}")

In [ ]:
for i, layer in enumerate(model_mob2.layers):
    if i < 155:
        layer.trainable = False
    else:
        layer.trainable = True

# Keep BN frozen
for layer in model_mob2.layers:
    if isinstance(layer, keras.layers.BatchNormalization):
        layer.trainable = False

trainable = sum(1 for l in model_mob2.layers if l.trainable)
frozen = sum(1 for l in model_mob2.layers if not l.trainable)
print(f"Trainable: {trainable}, Frozen: {frozen}")

model_mob2.compile(
    optimizer=keras.optimizers.Adam(5e-5),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=[
        keras.metrics.F1Score(average="macro", name="f1_score"),
        keras.metrics.TopKCategoricalAccuracy(k=3, name="top3_acc"),
    ],
)

history_mob2_phase2 = model_mob2.fit(
    train_ds,
    validation_data=val_ds,
    epochs=40,
    class_weight=class_weight_dict,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor="val_f1_score", patience=8,
            restore_best_weights=True, mode="max",
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_f1_score", factor=0.5,
            patience=4, min_lr=1e-7, mode="max", verbose=1,
        ),
        keras.callbacks.ModelCheckpoint(
            filepath="/kaggle/working/models/mobilenetv3_v2_phase2.keras",
            monitor="val_f1_score", save_best_only=True,
            mode="max", verbose=1,
        ),
    ],
    verbose=1,
)

best_epoch = int(np.argmax(history_mob2_phase2.history["val_f1_score"]))
val_f1     = float(history_mob2_phase2.history["val_f1_score"][best_epoch])
train_f1   = float(history_mob2_phase2.history["f1_score"][best_epoch])
stopped    = len(history_mob2_phase2.history["val_f1_score"])

print(f"\n  MobileNetV3Large v2 Phase 2")
print(f"  Val F1      : {val_f1:.4f}")
print(f"  Train F1    : {train_f1:.4f}")
print(f"  Gap         : {train_f1 - val_f1:.4f}")
print(f"  Stopped     : epoch {stopped}")

# Ensemble

In [ ]:
model_v2s = keras.models.load_model(
    "/kaggle/working/models/efficientnetv2s_phase3.keras"
)
model_mob = keras.models.load_model(
    "/kaggle/working/models/mobilenetv3_phase2.keras"
)

all_preds_v2s = []
all_preds_mob = []
all_labels    = []

for images, labels in val_ds:
    all_preds_v2s.append(model_v2s(images, training=False).numpy())
    all_preds_mob.append(model_mob(images, training=False).numpy())
    all_labels.append(labels.numpy())

preds_v2s  = np.concatenate(all_preds_v2s, axis=0)
preds_mob  = np.concatenate(all_preds_mob, axis=0)
all_labels = np.concatenate(all_labels, axis=0)

# Equal weight ensemble
preds_ensemble = (preds_v2s + preds_mob) / 2

def compute_f1(preds, labels):
    m = keras.metrics.F1Score(average="macro")
    m.update_state(labels, preds)
    return float(m.result())

f1_v2s     = compute_f1(preds_v2s, all_labels)
f1_mob     = compute_f1(preds_mob, all_labels)
f1_equal   = compute_f1(preds_ensemble, all_labels)

# Weighted ensemble favoring V2S
preds_weighted = (0.6 * preds_v2s + 0.4 * preds_mob)
f1_weighted = compute_f1(preds_weighted, all_labels)

print(f"V2S alone        : {f1_v2s:.4f}")
print(f"MobileNet alone  : {f1_mob:.4f}")
print(f"Ensemble (50/50) : {f1_equal:.4f}")
print(f"Ensemble (60/40) : {f1_weighted:.4f}")

# Deploy

In [ ]:
all_preds_v2s_test = []
all_preds_mob_test = []
all_labels_test    = []

for images, labels in test_ds:
    all_preds_v2s_test.append(model_v2s(images, training=False).numpy())
    all_preds_mob_test.append(model_mob(images, training=False).numpy())
    all_labels_test.append(labels.numpy())

preds_v2s_test  = np.concatenate(all_preds_v2s_test, axis=0)
preds_mob_test  = np.concatenate(all_preds_mob_test, axis=0)
all_labels_test = np.concatenate(all_labels_test, axis=0)

preds_ensemble_test = (preds_v2s_test + preds_mob_test) / 2

f1_test = compute_f1(preds_ensemble_test, all_labels_test)
print(f"Ensemble test F1 : {f1_test:.4f}")

pred_classes  = np.argmax(preds_ensemble_test, axis=1)
label_classes = np.argmax(all_labels_test, axis=1)
print(classification_report(label_classes, pred_classes,
                             target_names=class_names, digits=3))

In [ ]:
top3_metric = keras.metrics.TopKCategoricalAccuracy(k=3)
top3_metric.update_state(all_labels_test, preds_ensemble_test)
print(f"Top-3 accuracy: {float(top3_metric.result()):.4f}")

top2_metric = keras.metrics.TopKCategoricalAccuracy(k=2)
top2_metric.update_state(all_labels_test, preds_ensemble_test)
print(f"Top-2 accuracy: {float(top2_metric.result()):.4f}")

In [ ]:
from sklearn.metrics import cohen_kappa_score
kappa = cohen_kappa_score(label_classes, pred_classes)
print(f"Cohen's Kappa: {kappa:.4f}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(label_classes, pred_classes)

# Short names for readability
short_names = [
    "Durer", "Kustodiev", "Pissarro", "Hassam", "Monet",
    "Degas", "Boudin", "Dore", "Repin", "Aivazovsky",
    "Shishkin", "Sargent", "Chagall", "Saryan", "Roerich",
    "Picasso", "Cezanne", "Renoir", "Konchalovsky", "Kirchner",
    "Rembrandt", "Dali", "Van_Gogh"
]

fig, ax = plt.subplots(figsize=(18, 16))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=short_names,
    yticklabels=short_names,
    ax=ax,
    linewidths=0.5,
)
ax.set_xlabel("Predicted", fontsize=13)
ax.set_ylabel("True", fontsize=13)
ax.set_title("Confusion Matrix — Ensemble (50/50) Test Set", fontsize=15)
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig("/kaggle/working/confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
from sklearn.metrics import roc_auc_score
auc = roc_auc_score(all_labels_test, preds_ensemble_test,
                    multi_class="ovr", average="macro")
print(f"Macro ROC AUC: {auc:.4f}")

In [ ]:
from sklearn.metrics import matthews_corrcoef
mcc = matthews_corrcoef(label_classes, pred_classes)
print(f"MCC: {mcc:.4f}")

In [ ]:
confidences   = np.max(preds_ensemble_test, axis=1)
correct_mask  = (pred_classes == label_classes)

print(f"Mean confidence correct   : {confidences[correct_mask].mean():.4f}")
print(f"Mean confidence incorrect : {confidences[~correct_mask].mean():.4f}")

In [ ]:
# ── 1. Feature extractors (correct penultimate layers) ────────────────────────

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    extractor_v2s = tf.keras.Model(
        inputs  = model_v2s.input,
        outputs = model_v2s.get_layer("dropout").output       # (None, 1280)
    )
    extractor_mob = tf.keras.Model(
        inputs  = model_mob.input,
        outputs = model_mob.get_layer("dropout_2").output     # (None, 960)
    )

# ── 2. Helpers ────────────────────────────────────────────────────────────────

def _undo_preprocess(img_array):
    """Values already in [0, 255] float — just cast."""
    return np.clip(img_array, 0, 255).astype(np.uint8)

def _extract_feat(arr):
    """Batch or single image → L2-normalised concatenated feature vector."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        f_v2s = extractor_v2s(arr, training=False).numpy()   # (B, 1280)
        f_mob = extractor_mob(arr, training=False).numpy()   # (B, 960)
    feats  = np.concatenate([f_v2s, f_mob], axis=1)         # (B, 2240)
    feats /= (np.linalg.norm(feats, axis=1, keepdims=True) + 1e-8)
    return feats

# ── 3. Build train index (batch inference, not storing images in RAM) ───────────

def build_train_index(train_ds, class_names):
    all_labels, all_feats, all_refs = [], [], []

    for batch_idx, (images, labels) in enumerate(train_ds):
        feats      = _extract_feat(images)
        labels_np  = labels.numpy()
        label_ints = np.argmax(labels_np, axis=1) if labels_np.ndim == 2 else labels_np

        all_feats.append(feats.astype(np.float32))
        all_labels.extend([class_names[i] for i in label_ints])
        all_refs.extend([(batch_idx, i) for i in range(len(label_ints))])

        if batch_idx % 10 == 0:
            print(f"  Indexed {len(all_labels):,} training samples...")

    all_feats  = np.vstack(all_feats)
    all_labels = np.array(all_labels)
    all_refs   = np.array(all_refs)
    print(f"\nIndex ready: {len(all_labels):,} training samples.")
    return all_labels, all_feats, all_refs

# ── 4. Nearest-neighbour lookup restricted to predicted class ─────────────────

def find_nearest_train(feat, pred_cls, train_labels, train_feats, train_refs):
    mask      = train_labels == pred_cls
    sub_feats = train_feats[mask]
    sub_refs  = train_refs[mask]
    if len(sub_feats) == 0:
        return None, None, None
    cosine_dists        = 1 - sub_feats @ feat
    best                = np.argmin(cosine_dists)
    best_batch, best_sample = sub_refs[best]
    return int(best_batch), int(best_sample), float(cosine_dists[best])

# ── 5. Re-fetch one training image by its (batch_idx, sample_idx) ref ─────────

def fetch_train_image(train_ds, batch_idx, sample_idx):
    for i, (images, _) in enumerate(train_ds):
        if i == batch_idx:
            return _undo_preprocess(images.numpy()[sample_idx])
    return None

# ── 6. Plot ───────────────────────────────────────────────────────────────────

def _plot_pairs(misses, cols):
    if not misses:
        print("No misclassifications found.")
        return

    rows = (len(misses) + cols - 1) // cols
    fig, axes = plt.subplots(rows * 2, cols, figsize=(cols * 3.5, rows * 7))
    axes = axes.reshape(rows * 2, cols)

    for i, m in enumerate(misses):
        r, c = (i // cols) * 2, i % cols

        axes[r, c].imshow(m['test_img'])
        axes[r, c].set_title(
            f"TRUE: {m['true']}\n→ predicted: {m['pred']}",
            fontsize=8, color='firebrick', fontweight='bold'
        )
        axes[r, c].axis('off')

        axes[r+1, c].imshow(m['nn_img'])
        axes[r+1, c].set_title(
            f"NN train: {m['pred']}\ncosine dist: {m['nn_dist']:.3f}",
            fontsize=8, color='steelblue', fontweight='bold'
        )
        axes[r+1, c].axis('off')

    for i in range(len(misses), rows * cols):
        r, c = (i // cols) * 2, i % cols
        axes[r, c].axis('off')
        axes[r+1, c].axis('off')

    red  = mpatches.Patch(color='firebrick', label='Misclassified test image (true label)')
    blue = mpatches.Patch(color='steelblue', label='Most similar train image from predicted class')
    fig.legend(handles=[red, blue], loc='lower center', ncol=2, fontsize=9, frameon=False)
    plt.suptitle(
        "Misclassified test images — nearest train neighbour (ensemble features)",
        fontsize=11, y=1.01
    )
    plt.tight_layout()
    plt.savefig("misclassified_nn.png", dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved → misclassified_nn.png")

# ── 7. Main ───────────────────────────────────────────────────────────────────

def show_misclassified_nn(
    train_ds, class_names,
    preds_v2s_test, preds_mob_test, all_labels_test,
    test_ds,
    ensemble_weight=0.5, max_pairs=20, cols=4,
):
    print("Building training feature index...")
    train_labels, train_feats, train_refs = build_train_index(train_ds, class_names)

    preds_ensemble = (ensemble_weight       * preds_v2s_test +
                      (1 - ensemble_weight) * preds_mob_test)
    pred_classes  = np.argmax(preds_ensemble,  axis=1)
    label_classes = np.argmax(all_labels_test, axis=1)

    print("\nCollecting test images...")
    test_images_list = []
    for images, _ in test_ds:
        test_images_list.extend(images.numpy())

    print(f"Searching {len(pred_classes)} test samples for misclassifications...")
    misses = []
    for idx in range(len(pred_classes)):
        true_label = label_classes[idx]
        pred_label = pred_classes[idx]
        if true_label == pred_label:
            continue

        true_cls = class_names[true_label]
        pred_cls = class_names[pred_label]

        arr  = np.expand_dims(test_images_list[idx], 0)
        feat = _extract_feat(arr).squeeze()

        best_batch, best_sample, dist = find_nearest_train(
            feat, pred_cls, train_labels, train_feats, train_refs
        )
        if best_batch is None:
            continue

        nn_img = fetch_train_image(train_ds, best_batch, best_sample)

        misses.append({
            'test_img': _undo_preprocess(test_images_list[idx]),
            'true':     true_cls,
            'pred':     pred_cls,
            'nn_img':   nn_img,
            'nn_dist':  dist,
        })

        if len(misses) >= max_pairs:
            break

    print(f"Found {len(misses)} misclassified pairs. Plotting...")
    _plot_pairs(misses, cols)

# ── 8. Run ────────────────────────────────────────────────────────────────────

show_misclassified_nn(
    train_ds        = train_ds,
    class_names     = ds[0].class_names,
    preds_v2s_test  = preds_v2s_test,
    preds_mob_test  = preds_mob_test,
    all_labels_test = all_labels_test,
    test_ds         = test_ds,
    ensemble_weight = 0.5,
    max_pairs       = 20,
    cols            = 4,
)